# Bakken Den Haag
https://den-haag-opendata.opendatasoft.com/explore/assets/bakken/

Beschrijving: Locaties van ondergrondse en bovengrondse afval apartplaatsen (inclusief Oracs) en zoutafhaalkisten.
Bron: beheersysteem gemeente Den Haag
Doel registratie: Beheer openbare ruimte
Coördinatenstelsel: RDnew. De KML is WGS84

https://den-haag-opendata.opendatasoft.com/explore/assets/wijken/
Beschrijving: Wijkgrenzen van Den Haag
Bron: gemeente Den Haag
Doel registratie: Bestuurlijke grenzen
Coördinatenstelsel: RDnew. De KML is WGS84

## Data
- `bakken`: GeoDataFrame of Den Haag waste containers (CRS: WGS84/OGC:CRS84)
  - Key columns: `afvalfractie_code` (e.g. `"PPR"` = paper), `wijk_code` (str), `geometry` (points)
- `wijken`: GeoDataFrame of Den Haag neighbourhood borders
  - Key columns: `wijkcode` (int), `wijknaam` (str), `geometry` (polygons), `geo_point_2d` (centroids — default/active geometry, switch with `set_geometry('geometry')`)
  - CRS: OGC:CRS84

## Gotchas
- `wijken.geometry` is active but contains centroids; polygons are in the `geometry` column — always `set_geometry('geometry').set_crs('OGC:CRS84')` before use
- `wijken['wijkcode']` is `int64`; `bakken['wijk_code']` is `str`
- Reproject to EPSG:3857 for contextily basemaps and metric distances

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import shapely
import contextily as cx

In [ ]:
wijken = gpd.read_parquet('../data/wijken.parquet')
bakken = gpd.read_parquet('../data/bakken.parquet')

In [ ]:
# Filter with correct geometry column
wijken_poly = wijken.set_geometry('geometry').set_crs('OGC:CRS84')
laak = wijken_poly[wijken_poly['wijkcode'] == 38].to_crs(epsg=3857)
laak_poly = laak.union_all()

paper_wm = bakken[bakken['afvalfractie_code'] == "PPR"].to_crs(epsg=3857)

# Grid covering the wijk
xmin, ymin, xmax, ymax = laak.total_bounds
res = 300
gx = np.linspace(xmin, xmax, res)
gy = np.linspace(ymin, ymax, res)
gxx, gyy = np.meshgrid(gx, gy)

# Mask grid points outside the wijk polygon
mask = shapely.contains_xy(laak_poly, gxx.ravel(), gyy.ravel()).reshape(res, res)

# Distance using ALL paper bins
coords = np.array([[g.x, g.y] for g in paper_wm.geometry])
tree = cKDTree(coords)
dist, _ = tree.query(np.c_[gxx.ravel(), gyy.ravel()])
dist_grid = np.where(mask, dist.reshape(res, res), np.nan)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(
    dist_grid,
    origin='lower',
    extent=[xmin, xmax, ymin, ymax],
    cmap='RdYlGn_r',
    alpha=0.7,
    aspect='auto',
    zorder=2
)
laak.plot(ax=ax, color='none', edgecolor='black', linewidth=2, zorder=3)
paper_wm.cx[xmin:xmax, ymin:ymax].plot(ax=ax, color='blue', markersize=5, zorder=4, label='Paper bin')
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zorder=1)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
plt.colorbar(im, ax=ax, label='Distance to nearest paper bin (m)')
ax.set_title('Distance to nearest paper bin – Laakkwartier en Spoorwijk')
ax.set_axis_off()
ax.legend()
plt.tight_layout()
plt.show()